# Structural design: what identifies a response surface's parameters

A carryover's weights sum to one, so a dose held constant passes through it unchanged and the
data carry *no* information on the decay. Identification comes from temporal contrast.
`Schedule` builds the patterns; `contrast_score` — mean squared first difference over mean
squared dose, scale-free, `0` for a flat schedule and at most `4` — is the heuristic, and
`fisher_information` on the surface is the model-based verdict.

**The one-forward rule.** Nothing here re-implements the surface. The Jacobian's linear
columns are exactly the design matrix `surface.linearize` returns, and the nonlinear columns
are derivatives of `surface.forward` itself (jax when available with x64, central finite
differences otherwise). Then `FI = JᵀJ / noise_sd²`, and the Laplace posterior covariance
under independent Gaussian priors is `(diag(1/prior_sd²) + FI)⁻¹`.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    DerivativeMethod, FisherInformation, IdentifiabilityRidge, IdentifyingDesign, Pattern, Schedule,
    alternating, constant, contrast_score, design_to_identify, expected_posterior_sd,
    fisher_information, identifiability_ridge, pulse, ramp, random_switchback, ridge_of,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import Design, GeometricCarryover, HillKernel

from axiom.display import enable

enable();  # every axiom result renders itself from here on

In [ ]:
T = 12
schedules: list[Schedule] = [
    constant(T, 50.0),
    pulse(T, 80.0, 20.0, on=2, off=2),
    alternating(T, 80.0, 20.0),
    ramp(T, 0.0, 100.0),
    random_switchback(T, 80.0, 20.0, seed=0),
]
for s in schedules:
    p: Pattern = s.pattern
    print(f"{p:18s} mean={s.mean:5.1f} total={s.total:6.1f} contrast={contrast_score(s):.3f}  {s.doses[:6]}...")

## A Hill surface with geometric carryover

`axiom.sim.surface_world` builds the surface, its true parameters and a panel; the design math
only needs `world.surface` (a `SupportsForward`) and the data mapping `forward` reads.

In [ ]:
N_UNITS, NOISE_SD = 2, 0.5
truth = {"beta_a": 10.0, "alpha": 5.0, "k_a": 50.0, "s_a": 2.0, "lam_a": 0.5}
world = surface_world(
    n_units=N_UNITS, n_periods=T, treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    carryover=GeometricCarryover(max_lag=4), doses=DosePlan(scale=50.0),
    intercept="shared", truth=truth, noise_sd=NOISE_SD, seed=0,
)
method: DerivativeMethod = "finite"

def data_with(schedule: Schedule) -> dict[str, np.ndarray]:
    data = dict(world.data)
    data["a"] = schedule.as_grid(N_UNITS)
    return data

for s in schedules:
    fi = fisher_information(world.surface, data_with(s), world.theta, NOISE_SD, method=method)
    assert isinstance(fi, FisherInformation)
    i = fi.index("lam_a")
    print(f"{s.pattern:18s} info[lam_a]={fi.as_array()[i, i]:10.3f}  det={fi.det:12.4g}  singular={fi.singular}")

In [ ]:
flat = fisher_information(world.surface, data_with(constant(T, 50.0)), world.theta, NOISE_SD, method=method)
assert isinstance(flat, FisherInformation)
print("parameters:", flat.parameters, "| round-off columns:", flat.detail.get("round_off_columns"))
print("flat-prior covariance ->", flat.covariance().reason[:90], "...")
print("expected posterior sd (flat prior) ->", type(expected_posterior_sd(None, flat)).__name__)

In [ ]:
pulsed = fisher_information(world.surface, data_with(pulse(T, 80.0, 20.0, on=2, off=2)), world.theta, NOISE_SD, method=method)
assert isinstance(pulsed, FisherInformation)
priors = {"alpha": 5.0, "beta_a": 5.0, "k_a": 20.0, "s_a": 1.0, "lam_a": 0.3}
sd1 = expected_posterior_sd(priors, pulsed)
sd2 = expected_posterior_sd(priors, pulsed + pulsed)  # information adds over independent rows
assert isinstance(sd1, dict) and isinstance(sd2, dict)
for name in pulsed.parameters:
    print(f"{name:7s} prior {priors[name]:5.2f} -> posterior sd {sd1[name]:.4f} (x2 rows: {sd2[name]:.4f})")

## The identifiability ridge

The flattest direction of the correlation-form information matrix is the combination of
parameters the design moves least — for a saturating kernel, the `beta`/`k` equifinality
ridge. Pairwise flat-prior posterior correlations need a nonsingular matrix.

In [ ]:
ridge = identifiability_ridge(world.surface, data_with(pulse(T, 80.0, 20.0, on=2, off=2)), world.theta, NOISE_SD, pairs=(("beta_a", "k_a"),), method=method)
assert isinstance(ridge, IdentifiabilityRidge)
print("direction:", {k: round(v, 3) for k, v in ridge.direction.items()})
print("ridge parameters:", ridge.ridge_parameters, "| condition number:", round(ridge.condition_number, 1))
print("corr(beta_a, k_a) =", round(ridge.correlations[0], 3))
print("from the flat design:", ridge_of(flat).ridge_parameters, "| with pairs ->", type(ridge_of(flat, pairs=(("beta_a", "k_a"),))).__name__)

## Choosing rows to identify one parameter

`design_to_identify` picks `n` candidate rows (replicates allowed) minimizing the target's
expected posterior sd by point exchange. A surface with carryover is handed in as its
`steady_state()`, whose `forward` accepts independent rows.

In [ ]:
steady = world.surface.steady_state()
theta = {k: v for k, v in world.theta.items() if k != "lam_a"}
candidates = Design(treatments=("a",), points=((0.0,), (20.0,), (50.0,), (100.0,), (200.0,)), kind="grid")
out = design_to_identify(steady, candidates, theta, NOISE_SD, target="k_a", n=8,
                         prior_sds={"alpha": 5.0, "beta_a": 5.0, "k_a": 20.0, "s_a": 1.0}, seed=0, method=method)
assert isinstance(out, IdentifyingDesign)
print("chosen doses:", [p[0] for p in out.design.points])
print(f"expected sd of k_a: {out.expected_sd:.3f} (prior 20.0); all: { {k: round(v, 3) for k, v in out.expected_sds.items()} }")
print("one row under flat priors ->", type(design_to_identify(steady, candidates, theta, NOISE_SD, target="k_a", n=1, seed=0, method=method)).__name__)